In [ ]:
import numpy as np
from collections import namedtuple
from itertools import product
from runWidthCalc import getWidth
import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, as_completed


In [ ]:
def getLambdaUVFor(pts,lambdaUV0=1e8,nWorkers=4):
    """
    Get the UV scale for a list of (m1,m0,ctau) points in parameter space.
    """

    lambdaUVs = {}
    cache_width = {}
    m1m0_points = sorted(set((m1,m0) for m1,m0,_ in pts))
    print(f'Computing widths for {len(m1m0_points)} points with {nWorkers} workers...')
    if not os.path.isdir('lifetime_scan'):
        # Generate the MadGraph process folder once (serially), before the parallel runs
        pt = m1m0_points.pop(0)
        cache_width[pt] = getWidth(*pt, lambdaUV0)

    with ThreadPoolExecutor(max_workers=nWorkers) as executor:
        futures = {executor.submit(getWidth, m1, m0, lambdaUV0): (m1,m0) for m1,m0 in m1m0_points}
        for fut in tqdm.tqdm(as_completed(futures), total=len(futures)):
            key = futures[fut]
            try:
                cache_width[key] = fut.result()
            except Exception as e:
                print(f"Width calculation failed for (m1,m0)={key}: {e}")

    for m1, m0, ctau in pts:
        width0 = cache_width[(m1, m0)]
        width_required = 1.97327e-16/ctau  # Convert ctau from meters to GeV       
        lambdaUV = lambdaUV0*np.sqrt(width0/width_required)
        lambdaUVs[(m1,m0,ctau)] = lambdaUV

    return lambdaUVs
    

## New scan grid

In [ ]:

mSs = [300.,500.,700.,900.,1100.] #mass of scalar particle [GeV]
dmSs =[10.,50.,100.,200.,400.] # mS - (m1+m2)
dm12 = 10.0
dm10s = [15.0,50.,100.,200.,400.]
ctaus = [10,20,50] #proper lifetime of Chi1 [m]
lambdaUV0 = 1e8
nWorkers = os.cpu_count() # number of parallel MadEvent runs

# 1) Build the list of valid points (no width computation yet)
grid = []
for mS, dmS, dm10, ctau in product(mSs, dmSs, dm10s, ctaus):
    m2 = float(round(((mS - dmS) + dm12)/2.,2))
    m1 = float(round(m2 - dm12,2))
    m0 = float(round(m1-dm10,2))
    if any(m < 0 for m in [mS, m1, m2, m0]):
        continue
    if sorted([m0,m1,m2,mS]) != [m0,m1,m2,mS]:
        continue
    if m1-m0 >= 250.: # Exclude points which allow chi1 -> chi0 + h + h
        continue
    grid.append((mS, m2, m1, m0, ctau))

# 2) Compute the widths for the unique (m1,m0) pairs in parallel.
# Each getWidth call runs MadEvent in its own temporary copy of the process folder,
# so the calls are independent. Threads are enough, since the work is done in a subprocess.
pts = [(m1,m0,ctau) for _,_,m1,m0,ctau in grid]
lambdaUVs = getLambdaUVFor(pts, lambdaUV0=lambdaUV0, nWorkers=5)

# 3) Build the points
pts = []
for mS, m2, m1, m0, ctau in grid:
    lambdaUV = lambdaUVs[(m1,m0,ctau)]
    pt = [f"{x:1.4g}" if x < 1e4 else f"{x:1.4e}" for x in (mS, m2, m1, m0, lambdaUV, ctau)]
    pts.append(tuple(pt))

print(len(pts))

100%|██████████| 77/77 [17:38<00:00, 13.75s/it] 

267


In [4]:
print(str(pts).replace("'",""))

[(300, 150, 140, 125, 1.0121e+05, 10), (300, 150, 140, 125, 1.4313e+05, 20), (300, 150, 140, 125, 2.2631e+05, 50), (300, 150, 140, 90, 3.5995e+06, 10), (300, 150, 140, 90, 5.0904e+06, 20), (300, 150, 140, 90, 8.0487e+06, 50), (300, 150, 140, 40, 2.0208e+07, 10), (300, 150, 140, 40, 2.8579e+07, 20), (300, 150, 140, 40, 4.5187e+07, 50), (300, 130, 120, 105, 9.9708e+04, 10), (300, 130, 120, 105, 1.4101e+05, 20), (300, 130, 120, 105, 2.2295e+05, 50), (300, 130, 120, 70, 3.3748e+06, 10), (300, 130, 120, 70, 4.7728e+06, 20), (300, 130, 120, 70, 7.5464e+06, 50), (300, 130, 120, 20, 1.5188e+07, 10), (300, 130, 120, 20, 2.1479e+07, 20), (300, 130, 120, 20, 3.3961e+07, 50), (300, 105, 95, 80, 9.6948e+04, 10), (300, 105, 95, 80, 1.3710e+05, 20), (300, 105, 95, 80, 2.1678e+05, 50), (300, 105, 95, 45, 2.9468e+06, 10), (300, 105, 95, 45, 4.1675e+06, 20), (300, 105, 95, 45, 6.5893e+06, 50), (300, 55, 45, 30, 8.2121e+04, 10), (300, 55, 45, 30, 1.1614e+05, 20), (300, 55, 45, 30, 1.8363e+05, 50), (500, 

# Jack's Grid

In [39]:
mSs = [400, 450, 600, 800, 1000] #mass of scalar particle [GeV]
# beta1s = [0.1, 0.2, 0.25, 0.3, 0.4] #boost of the chi1 in the labframe
beta1s = [0.1,0.2,0.25,0.3,0.4] #boost of the chi1 in the labframe
Dm21s = [5] #mass difference between chi2 and chi1 [GeV]
Dm10s = [15,30,60,100] #mass difference between chi1 and chi0 [GeV] (come back and try 100)
ctau0s = [20] #proper lifetime of Chi1 [m]

In [40]:
pts_scan_ref = []
for mS,beta1,Dm21,Dm10,ctau0 in product(mSs,beta1s,Dm21s,Dm10s,ctau0s):
    m1_solutions = [ (-beta1 ** 2 * Dm21 ** 3 + sgn * (-beta1 ** 2 * Dm21 ** 4 * mS ** 2 + 2 * beta1 ** 2 * Dm21 ** 2 * mS ** 4 - beta1 ** 2 * mS ** 6 + Dm21 ** 4 * mS ** 2 - 2 * Dm21 ** 2 * mS ** 4 + mS ** 6) ** .5 + beta1 ** 2 * Dm21 * mS ** 2 + Dm21 ** 3 - Dm21 * mS ** 2)/(2 * (beta1 ** 2 * Dm21 ** 2 - Dm21 ** 2 + mS ** 2)) for sgn in [+1 , -1]]
    M1 = max(m1_solutions)
    M2 = M1 + Dm21 #mass of chi2 [GeV]
    M0 = M1 - Dm10 #mass of chi0 [GeV]
    MS = mS
    if sorted([M0,M1,M2,MS]) != [M0,M1,M2,MS]:
        continue
    if M1-M0 >= 250.: # Exclude points which allow chi1 -> chi0 + h + h
        continue
    pts_scan_ref.append((MS,M2,M1,M0,1e8,ctau0))

m1m0_ctau_points = sorted(set((m1,m0,ctau) for _,_,m1,m0,_,ctau in pts_scan_ref))
lambdaUVs_scan = getLambdaUVFor(m1m0_ctau_points, lambdaUV0=1e8, nWorkers=5)
pts_scan = []
for pt in pts_scan_ref:
    MS,M2,M1,M0,_,ctau0 = pt
    lambdaUV = lambdaUVs_scan[(M1,M0,ctau0)]
    pt_new = [f"{x:1.4g}" if x < 1e4 else f"{x:1.4e}" for x in (MS,M2,M1,M0,lambdaUV,ctau0)]
    pts_scan.append(tuple(pt_new))

Coimputing widths for 100 with 5 workers...


100%|██████████| 100/100 [15:48<00:00,  9.49s/it]


In [42]:
print(str(pts_scan).replace("'",""))

[(400, 201.5, 196.5, 181.5, 1.4679e+05, 20), (400, 201.5, 196.5, 166.5, 1.4408e+06, 20), (400, 201.5, 196.5, 136.5, 9.0757e+06, 20), (400, 201.5, 196.5, 96.52, 3.9846e+07, 20), (400, 198.6, 193.6, 178.6, 1.4674e+05, 20), (400, 198.6, 193.6, 163.6, 1.4370e+06, 20), (400, 198.6, 193.6, 133.6, 9.0395e+06, 20), (400, 198.6, 193.6, 93.56, 3.9422e+07, 20), (400, 196.3, 191.3, 176.3, 1.4660e+05, 20), (400, 196.3, 191.3, 161.3, 1.4366e+06, 20), (400, 196.3, 191.3, 131.3, 9.0062e+06, 20), (400, 196.3, 191.3, 91.3, 3.9182e+07, 20), (400, 193.5, 188.5, 173.5, 1.4641e+05, 20), (400, 193.5, 188.5, 158.5, 1.4334e+06, 20), (400, 193.5, 188.5, 128.5, 8.9419e+06, 20), (400, 193.5, 188.5, 88.51, 3.8744e+07, 20), (400, 186.2, 181.2, 166.2, 1.4582e+05, 20), (400, 186.2, 181.2, 151.2, 1.4248e+06, 20), (400, 186.2, 181.2, 121.2, 8.8478e+06, 20), (400, 186.2, 181.2, 81.2, 3.7635e+07, 20), (450, 226.4, 221.4, 206.4, 1.4775e+05, 20), (450, 226.4, 221.4, 191.4, 1.4621e+06, 20), (450, 226.4, 221.4, 161.4, 9.3894